# 面试问题：怎样保证训练可复现，并让 Checkpoint Resume 与不中断训练一致？

可以直接复述的回答是：只设置一次 seed 不等于可复现，因为参数初始化、样本 shuffle、dropout、数据增强和并行 kernel 都会消费随机状态。一个可继续训练的 checkpoint 至少应保存模型参数、优化器状态、epoch/step、学习率调度器、混合精度 scaler 和所有随机数生成器状态。恢复顺序也重要：先重建对象，再加载状态，最后从准确的下一个 batch 继续。验证 resume 的方法是从同一初值分别做不中断训练和中断恢复训练，并逐参数比较。下面不用 `torch.optim`，手写带 Momentum 的小批训练，真实保存和恢复随机状态。

## 真实案例：配送延迟风险二分类训练恢复

十条脱敏配送记录包含距离、雨量、仓库拥堵和延迟标签。模型含 ReLU 与 dropout，训练还包含随机 shuffle，因此只恢复权重一定会走向不同参数。小数据仅用于证明状态机，不代表线上预测能力。

In [1]:
import io  # 导入内存二进制流用于真实 checkpoint 序列化
import random  # 导入 Python 随机状态管理
import warnings  # 导入告警控制模块
import numpy as np  # 导入 NumPy 随机状态管理
import torch  # 导入 PyTorch 张量与自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警以保持输出聚焦
torch.set_num_threads(1)  # 固定 CPU 单线程执行减少非确定性
random.seed(12)  # 固定 Python 初始随机状态
np.random.seed(12)  # 固定 NumPy 初始随机状态
torch.manual_seed(12)  # 固定 PyTorch 参数初始化
records = [  # 定义十条带业务语义的配送记录
    ("E-01", 2.0, 0.0, 0.1, 0.0),  # 近距离晴天低拥堵准时件
    ("E-02", 8.0, 0.2, 0.3, 0.0),  # 中距离轻微降雨准时件
    ("E-03", 15.0, 0.8, 0.7, 1.0),  # 长距离大雨拥堵延迟件
    ("E-04", 5.0, 0.9, 0.8, 1.0),  # 近距离但极端天气延迟件
    ("E-05", 12.0, 0.1, 0.2, 0.0),  # 长距离低风险准时件
    ("E-06", 18.0, 0.6, 0.4, 1.0),  # 长距离中雨延迟件
    ("E-07", 3.0, 0.3, 0.9, 1.0),  # 仓库严重拥堵延迟件
    ("E-08", 9.0, 0.0, 0.1, 0.0),  # 晴天低拥堵准时件
    ("E-09", 20.0, 0.7, 0.9, 1.0),  # 多因素高风险延迟件
    ("E-10", 6.0, 0.1, 0.2, 0.0),  # 普通短途准时件
]  # 结束十条配送记录
x = torch.tensor([[row[1] / 20.0, row[2], row[3]] for row in records], dtype=torch.float32)  # 缩放距离并构造三维输入
y = torch.tensor([row[4] for row in records], dtype=torch.float32)  # 构造二分类延迟标签
print("输入预览：id | 距离km | 雨量 | 拥堵 | 延迟")  # 输出原始配送字段标题
for row in records:  # 逐条展示十个配送样本
    print(f"{row[0]} | {row[1]:5.1f} | {row[2]:.1f} | {row[3]:.1f} | {int(row[4])}")  # 展示训练输入和监督标签
print("随机来源：参数初始化 + 每轮 shuffle + 每批 dropout")  # 明确 checkpoint 必须覆盖的状态

输入预览：id | 距离km | 雨量 | 拥堵 | 延迟
E-01 |   2.0 | 0.0 | 0.1 | 0
E-02 |   8.0 | 0.2 | 0.3 | 0
E-03 |  15.0 | 0.8 | 0.7 | 1
E-04 |   5.0 | 0.9 | 0.8 | 1
E-05 |  12.0 | 0.1 | 0.2 | 0
E-06 |  18.0 | 0.6 | 0.4 | 1
E-07 |   3.0 | 0.3 | 0.9 | 1
E-08 |   9.0 | 0.0 | 0.1 | 0
E-09 |  20.0 | 0.7 | 0.9 | 1
E-10 |   6.0 | 0.1 | 0.2 | 0
随机来源：参数初始化 + 每轮 shuffle + 每批 dropout


## Baseline / 基线：Checkpoint 只保存 model weights

我们先完成 6 轮不中断参考训练，再在第 3 轮只保存权重。恢复时 Momentum 清零且随机生成器换 seed，即使数据和代码相同，后 3 轮也无法复现参考轨迹。

In [2]:
class DelayNet(torch.nn.Module):  # 定义含随机 dropout 的小型延迟分类器
    def __init__(self):  # 初始化两层可训练参数
        super().__init__()  # 初始化 PyTorch 模块基类
        self.weight1 = torch.nn.Parameter(torch.randn(3, 5) * 0.20)  # 创建输入到五维隐层权重
        self.bias1 = torch.nn.Parameter(torch.zeros(5))  # 创建隐层偏置
        self.weight2 = torch.nn.Parameter(torch.randn(5) * 0.20)  # 创建隐层到单 logit 权重
        self.bias2 = torch.nn.Parameter(torch.tensor(0.0))  # 创建输出偏置
    def forward(self, features, generator, use_dropout):  # 定义显式接收随机生成器的前向传播
        hidden = torch.relu(features @ self.weight1 + self.bias1)  # 计算确定性 ReLU 隐层
        if use_dropout:  # 只在训练路径应用 dropout
            mask = (torch.rand(hidden.shape, generator=generator) >= 0.25).float() / 0.75  # 用可保存生成器采样 inverted dropout mask
            hidden = hidden * mask  # 应用随机掩码并保持期望激活尺度
        return hidden @ self.weight2 + self.bias2  # 计算每条配送记录延迟 logit
def binary_cross_entropy(logits, labels):  # 手写数值稳定二元交叉熵
    return (torch.clamp(logits, min=0.0) - logits * labels + torch.log1p(torch.exp(-logits.abs()))).mean()  # 返回小批量平均损失
def fresh_velocity(model):  # 为手写 Momentum 创建零速度状态
    return {name: torch.zeros_like(parameter) for name, parameter in model.named_parameters()}  # 按参数名建立可序列化速度张量
def train_epochs(model, velocity, generator, start_epoch, end_epoch):  # 执行可中断的小批量训练区间
    history = []  # 保存每轮顺序、损失和参数摘要
    for epoch in range(start_epoch, end_epoch):  # 从指定 epoch 恢复到目标 epoch
        order = torch.randperm(len(x), generator=generator)  # 用显式生成器产生本轮样本顺序
        for batch_start in range(0, len(x), 2):  # 按两个样本组成 mini-batch
            indices = order[batch_start:batch_start + 2]  # 读取当前随机小批索引
            logits = model(x[indices], generator, True)  # 真实执行含 dropout 的 forward
            loss = binary_cross_entropy(logits, y[indices])  # 计算当前小批分类损失
            loss.backward()  # 真实执行 backward 得到参数梯度
            with torch.no_grad():  # 关闭手写 Momentum 更新计算图
                for name, parameter in model.named_parameters():  # 按名称遍历全部参数
                    velocity[name] = 0.8 * velocity[name] + parameter.grad  # 更新可恢复的一阶速度状态
                    parameter -= 0.30 * velocity[name]  # 使用速度执行参数更新并在六轮内形成可观察学习效果
                    parameter.grad.zero_()  # 清空当前小批梯度
        with torch.no_grad():  # 使用无 dropout 路径评估本轮模型
            epoch_loss = binary_cross_entropy(model(x, generator, False), y)  # 计算完整数据确定性损失
            parameter_sum = sum(float(parameter.sum()) for parameter in model.parameters())  # 形成便于观察的参数摘要
        history.append({"epoch": epoch + 1, "order": order.tolist(), "loss": float(epoch_loss), "parameter_sum": parameter_sum})  # 保存本轮真实训练轨迹
    return history  # 返回指定训练区间的历史
template_model = DelayNet()  # 创建所有实验共享的初始模型
initial_state = {name: value.detach().clone() for name, value in template_model.state_dict().items()}  # 深拷贝公平比较初始参数
full_model = DelayNet()  # 创建不中断参考模型
full_model.load_state_dict(initial_state)  # 恢复共享初始参数
full_velocity = fresh_velocity(full_model)  # 初始化参考 Momentum 状态
full_generator = torch.Generator().manual_seed(2026)  # 创建参考 shuffle 与 dropout 随机流
full_history = train_epochs(full_model, full_velocity, full_generator, 0, 6)  # 连续训练六轮作为权威轨迹
bad_partial_model = DelayNet()  # 创建只保存权重的中断实验模型
bad_partial_model.load_state_dict(initial_state)  # 恢复相同初始参数
bad_partial_velocity = fresh_velocity(bad_partial_model)  # 初始化中断实验 Momentum 状态
bad_partial_generator = torch.Generator().manual_seed(2026)  # 使用与参考相同的起始随机流
bad_first_history = train_epochs(bad_partial_model, bad_partial_velocity, bad_partial_generator, 0, 3)  # 训练前三轮后模拟中断
weights_only = {name: value.detach().clone() for name, value in bad_partial_model.state_dict().items()}  # 基线 checkpoint 只复制模型权重
bad_resumed_model = DelayNet()  # 重建错误恢复模型
bad_resumed_model.load_state_dict(weights_only)  # 仅加载模型权重而遗漏其他状态
bad_resumed_velocity = fresh_velocity(bad_resumed_model)  # 错误地把 Momentum 速度清零
bad_resumed_generator = torch.Generator().manual_seed(999)  # 错误地从新随机流继续 shuffle 和 dropout
bad_second_history = train_epochs(bad_resumed_model, bad_resumed_velocity, bad_resumed_generator, 3, 6)  # 完成错误恢复后的后三轮
bad_distance = float(torch.sqrt(sum((left - right).square().sum() for left, right in zip(full_model.parameters(), bad_resumed_model.parameters()))))  # 计算错误恢复与不中断参数距离
print("不中断训练：epoch | order前5 | loss | parameter_sum")  # 输出权威训练轨迹表头
for row in full_history:  # 遍历六轮不中断历史
    print(f"{row['epoch']} | {row['order'][:5]} | {row['loss']:.5f} | {row['parameter_sum']:.6f}")  # 展示随机顺序和参数轨迹
print(f"只恢复权重后的参数距离={bad_distance:.8f}")  # 展示基线恢复无法重现不中断训练

不中断训练：epoch | order前5 | loss | parameter_sum
1 | [5, 1, 4, 8, 7] | 0.69296 | -0.848124
2 | [4, 9, 2, 8, 5] | 0.69136 | -0.786596
3 | [3, 4, 0, 6, 5] | 0.69303 | -0.244632
4 | [2, 0, 4, 5, 9] | 0.63996 | 0.288680
5 | [7, 5, 8, 4, 2] | 0.45894 | 1.365052
6 | [2, 6, 3, 0, 4] | 0.33667 | 1.559476
只恢复权重后的参数距离=1.55132937


## 核心实现：保存模型、Momentum、进度与全部 RNG state

checkpoint 真实写入内存二进制流，再创建新对象并加载。这里训练只主动使用独立 `torch.Generator`，仍额外保存 Python、NumPy 和全局 Torch RNG，展示生产 checkpoint 的完整边界。

In [3]:
good_partial_model = DelayNet()  # 创建正确中断恢复实验模型
good_partial_model.load_state_dict(initial_state)  # 恢复共享初始参数
good_partial_velocity = fresh_velocity(good_partial_model)  # 初始化正确实验 Momentum 状态
good_partial_generator = torch.Generator().manual_seed(2026)  # 使用与不中断训练相同随机流
good_first_history = train_epochs(good_partial_model, good_partial_velocity, good_partial_generator, 0, 3)  # 训练相同前三轮
checkpoint = {  # 构造包含训练状态机的完整 checkpoint
    "model": {name: value.detach().clone() for name, value in good_partial_model.state_dict().items()},  # 保存模型参数和 buffer
    "optimizer_velocity": {name: value.detach().clone() for name, value in good_partial_velocity.items()},  # 保存手写 Momentum 速度
    "epoch": 3,  # 保存准确的下一恢复轮次
    "train_generator": good_partial_generator.get_state(),  # 保存 shuffle 与 dropout 专用随机流
    "python_rng": random.getstate(),  # 保存 Python 随机状态
    "numpy_rng": np.random.get_state(),  # 保存 NumPy 随机状态
    "torch_rng": torch.get_rng_state(),  # 保存全局 PyTorch 随机状态
}  # 结束完整 checkpoint 字典
buffer = io.BytesIO()  # 创建内存 checkpoint 文件
torch.save(checkpoint, buffer)  # 真实序列化模型、优化器、进度与 RNG 状态
checkpoint_size = buffer.tell()  # 记录序列化后的字节大小
buffer.seek(0)  # 把读指针移动到 checkpoint 开头
restored = torch.load(buffer, weights_only=False)  # 从二进制流真实反序列化训练状态
good_resumed_model = DelayNet()  # 重建全新的恢复模型对象
good_resumed_model.load_state_dict(restored["model"])  # 加载保存的模型状态
good_resumed_velocity = {name: value.detach().clone() for name, value in restored["optimizer_velocity"].items()}  # 恢复每个参数的 Momentum 速度
good_resumed_generator = torch.Generator()  # 创建待恢复的训练随机生成器
good_resumed_generator.set_state(restored["train_generator"])  # 恢复 shuffle 与 dropout 随机流位置
random.setstate(restored["python_rng"])  # 恢复 Python 随机状态
np.random.set_state(restored["numpy_rng"])  # 恢复 NumPy 随机状态
torch.set_rng_state(restored["torch_rng"])  # 恢复全局 PyTorch 随机状态
good_second_history = train_epochs(good_resumed_model, good_resumed_velocity, good_resumed_generator, restored["epoch"], 6)  # 从准确 epoch 完成后三轮
good_distance = float(torch.sqrt(sum((left - right).square().sum() for left, right in zip(full_model.parameters(), good_resumed_model.parameters()))))  # 计算正确恢复与不中断参数距离
print("checkpoint keys：", sorted(checkpoint.keys()))  # 展示实际保存的训练状态组成
print(f"checkpoint bytes={checkpoint_size}")  # 展示真实序列化结果大小
print("恢复后三轮：epoch | order前5 | loss | parameter_sum")  # 输出恢复轨迹表头
for row in good_second_history:  # 遍历正确恢复后的三轮历史
    print(f"{row['epoch']} | {row['order'][:5]} | {row['loss']:.5f} | {row['parameter_sum']:.6f}")  # 展示顺序与不中断轨迹完全一致
print(f"完整状态恢复后的参数距离={good_distance:.12f}")  # 展示位级一致的恢复结果

checkpoint keys： ['epoch', 'model', 'numpy_rng', 'optimizer_velocity', 'python_rng', 'torch_rng', 'train_generator']
checkpoint bytes=21304
恢复后三轮：epoch | order前5 | loss | parameter_sum
4 | [2, 0, 4, 5, 9] | 0.63996 | 0.288680
5 | [7, 5, 8, 4, 2] | 0.45894 | 1.365052
6 | [2, 6, 3, 0, 4] | 0.33667 | 1.559476
完整状态恢复后的参数距离=0.000000000000


## 逐样本结果：不中断、错误恢复与正确恢复

In [4]:
evaluation_generator = torch.Generator().manual_seed(1)  # 创建评估占位生成器但关闭 dropout
with torch.no_grad():  # 进入确定性评估阶段
    full_probability = torch.sigmoid(full_model(x, evaluation_generator, False))  # 计算不中断参考概率
    bad_probability = torch.sigmoid(bad_resumed_model(x, evaluation_generator, False))  # 计算只恢复权重后的概率
    good_probability = torch.sigmoid(good_resumed_model(x, evaluation_generator, False))  # 计算完整恢复后的概率
full_prediction = (full_probability >= 0.5).float()  # 形成不中断参考类别
full_accuracy = float((full_prediction == y).float().mean())  # 计算教学数据分类准确率
print("id | 标签 | 不中断概率 | 只权重恢复 | 完整恢复 | good差值")  # 输出逐样本恢复对照表头
for index, row in enumerate(records):  # 遍历十条配送记录
    difference = abs(float(good_probability[index] - full_probability[index]))  # 计算正确恢复逐样本概率差
    print(f"{row[0]} | {int(y[index])} | {full_probability[index]:10.6f} | {bad_probability[index]:10.6f} | {good_probability[index]:10.6f} | {difference:.2e}")  # 展示错误与正确恢复差异
print(f"教学集 accuracy={full_accuracy:.1%}，bad parameter distance={bad_distance:.8f}，good distance={good_distance:.12f}")  # 汇总模型质量与恢复一致性

id | 标签 | 不中断概率 | 只权重恢复 | 完整恢复 | good差值
E-01 | 0 |   0.286225 |   0.466247 |   0.286225 | 0.00e+00
E-02 | 0 |   0.291387 |   0.611182 |   0.291387 | 0.00e+00
E-03 | 1 |   0.815466 |   0.841976 |   0.815466 | 0.00e+00
E-04 | 1 |   0.815720 |   0.818026 |   0.815720 | 0.00e+00
E-05 | 0 |   0.286382 |   0.600664 |   0.286382 | 0.00e+00
E-06 | 1 |   0.626707 |   0.791147 |   0.626707 | 0.00e+00
E-07 | 1 |   0.522643 |   0.697232 |   0.522643 | 0.00e+00
E-08 | 0 |   0.283820 |   0.528589 |   0.283820 | 0.00e+00
E-09 | 1 |   0.860917 |   0.868610 |   0.860917 | 0.00e+00
E-10 | 0 |   0.288453 |   0.548399 |   0.288453 | 0.00e+00
教学集 accuracy=100.0%，bad parameter distance=1.55132937，good distance=0.000000000000


## 失败案例与修正：Dropout RNG 位置没有恢复

即使模型权重完全相同，连续两次训练态 forward 会消费不同随机 mask。把 generator state 恢复到 forward 前的位置，才能逐值重放同一个 dropout 输出。

In [5]:
probe_generator = torch.Generator().manual_seed(31415)  # 创建独立 dropout 重放随机流
probe_state = probe_generator.get_state()  # 保存第一次 mask 采样前随机位置
with torch.no_grad():  # 关闭探针前向梯度记录
    probe_hidden = torch.relu(x @ full_model.weight1 + full_model.bias1)  # 计算十条样本确定性隐层
    first_mask = (torch.rand(probe_hidden.shape, generator=probe_generator) >= 0.25).float() / 0.75  # 消费随机数生成第一组 dropout mask
    first_dropout_output = (probe_hidden * first_mask) @ full_model.weight2 + full_model.bias2  # 使用第一组 mask 计算训练态输出
    second_mask = (torch.rand(probe_hidden.shape, generator=probe_generator) >= 0.25).float() / 0.75  # 不恢复状态直接生成第二组 mask
    second_dropout_output = (probe_hidden * second_mask) @ full_model.weight2 + full_model.bias2  # 使用第二组 mask 计算不同训练态输出
probe_generator.set_state(probe_state)  # 修正方案恢复第一次采样前 RNG 状态
with torch.no_grad():  # 关闭重放计算的梯度记录
    replayed_mask = (torch.rand(probe_hidden.shape, generator=probe_generator) >= 0.25).float() / 0.75  # 重放第一组 dropout mask
    replayed_dropout_output = (probe_hidden * replayed_mask) @ full_model.weight2 + full_model.bias2  # 用重放 mask 重新计算输出
unrestored_mask_difference = int((second_mask != first_mask).sum())  # 统计未恢复 RNG 时不同的 mask 单元数
unrestored_difference = float(torch.linalg.vector_norm(second_dropout_output - first_dropout_output))  # 量化未恢复 RNG 的输出差异
replayed_difference = float(torch.linalg.vector_norm(replayed_dropout_output - first_dropout_output))  # 量化恢复 RNG 后的输出差异
print("前两条第一次 dropout 输出：", first_dropout_output[:2].tolist())  # 展示原始随机前向结果片段
print("前两条未恢复 RNG 输出：", second_dropout_output[:2].tolist())  # 展示相同权重的下一随机结果片段
print("前两条恢复 RNG 重放：", replayed_dropout_output[:2].tolist())  # 展示逐值重现的修正结果片段
print(f"未恢复 mask 差异单元={unrestored_mask_difference}，输出差值={unrestored_difference:.8f}，恢复后差值={replayed_difference:.8f}")  # 对比 RNG 状态门禁效果

前两条第一次 dropout 输出： [-0.9491816759109497, -0.9156710505485535]
前两条未恢复 RNG 输出： [-0.9491816759109497, -0.8076081871986389]
前两条恢复 RNG 重放： [-0.9491816759109497, -0.9156710505485535]
未恢复 mask 差异单元=27，输出差值=3.75538540，恢复后差值=0.00000000


## 结果解读

只保存 model weights 时，第四轮的 shuffle 顺序、dropout mask 和 Momentum 方向同时改变，最终参数明显偏离不中断训练。完整 checkpoint 恢复后三轮的顺序、损失和参数摘要与参考轨迹一致，最终距离为零。这里的“可复现”是同环境的受控实验，不等于跨 GPU、驱动和框架版本必然逐位一致。

## 生产边界

真实训练还要保存 scheduler、AMP GradScaler、全局 step、sampler epoch、数据游标、梯度累积位置和分布式各 rank RNG。checkpoint 写入应原子化并带校验和，恢复后先核对数据版本、模型代码、world size 与超参数。某些 CUDA kernel 仍可能非确定；启用 deterministic algorithm 会牺牲速度或缺少算子支持。

## 最小回归测试

In [6]:
assert len(records) >= 5  # 保证恢复实验使用多个可读业务样本
assert bad_distance > 1e-6  # 保证只恢复权重的失败确实偏离不中断训练
assert good_distance == 0.0  # 保证完整状态恢复与不中断训练逐参数一致
assert good_second_history == full_history[3:]  # 保证恢复后的顺序、损失和参数轨迹完全一致
assert torch.equal(good_probability, full_probability)  # 保证逐样本推理结果完全一致
assert unrestored_mask_difference > 0 and replayed_difference == 0.0  # 保证 dropout RNG 失败与修正均可复现
assert full_accuracy >= 0.7  # 保证真实训练得到基本可用的教学分类器